# 🤖 Telegram Eligibility Verification Bot

## Overview

This notebook contains the development and testing environment for a Telegram bot that performs eligibility and record verification using a National ID number (DNI).

The bot retrieves information from a structured dataset, validates user input, searches records efficiently, and generates consolidated verification reports through a conversational Telegram interface.

---

## 🎯 Objectives

- Automate administrative verification workflows
- Reduce manual lookup time
- Provide a user-friendly Telegram interface
- Consolidate information from multiple verification sources
- Enable rapid deployment using cloud or local environments

---

## ⚙️ Main Features

- DNI-based record search
- Input validation
- Automatic dataset refresh
- Multi-source verification reporting
- Telegram Bot integration
- CSV / Google Sheets backend
- Structured report generation

---


## Author

Martin Bielke

In [ ]:
# -*- coding: utf-8 -*-
# BOT DE VERIFICACIÓN PREVISIONAL - CONSULTA POR DNI (VERSIÓN STANDALONE)
# Fuente: Google Sheets (CSV público)
# =====================================================

import os
import re
import time
import requests
from io import StringIO
import pandas as pd
from dotenv import load_dotenv
from telegram import Update
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters,
)


load_dotenv()

# =========================
# CONFIGURACIÓN
# =========================
TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")

if not TELEGRAM_TOKEN:
    raise RuntimeError(
        "La variable de entorno TELEGRAM_TOKEN no está configurada."
    )


CSV_URL = os.getenv("CSV_URL")
LOCAL_CSV = "sample_data.csv"
    
    
REFRESH_INTERVAL = 300  # 5 minutos
BOT_NAME = "Eligibility Verification Bot"

# Columnas (según tu archivo)
SISTEMAS = [
    ("REGISTRA OBSERVACIONES EN LA NEGATIVA DE ANSES", "OBSERVACIONES DE LA NEGATIVA DE ANSES", "ANSES"),
    ("REGISTRA OBSERVACIONES EN LA SUPERINTENDENCIA DE SERVICIOS DE SALUD", "OBSERVACIONES DE LA SUPERINTENDENCIA DE SERVICIOS DE SALUD", "Superintendencia"),
    ("REGISTRA OBSERVACIONES EN MONOTRIBUTO DE LA SUPERINTENDENCIA DE SERVICIOS DE SALUD", "OBSERVACIONES EN MONOTRIBUTO DE LA SUPERINTENDENCIA DE SERVICIOS DE SALUD", "Monotributo"),
    ("REGISTRA OBSERVACIONES EN PUCO (sistema integrado de información sanitaria argentino)", "OBSERVACIONES EN PUCO (sistema integrado de información sanitaria argentino)", "PUCO"),
    ("REGISTRA OBSERVACIONES EN SINTYS (Sistema de Identificación Tributaria y Social)", "OBSERVACIONES EN SINTYS (Sistema de Identificación Tributaria y Social)", "SINTYS"),
    ("REGISTRA OBSERVACIONES EN ARCA –aportes en linea-", "OBSERVACIONES EN ARCA –aportes en linea-", "ARCA"),
    ("REGISTRA OBSERVACIONES EN -IPSS", "OBSERVACIONES EN IPSS", "IPSS"),
]

COL_NOMBRE = "NOMBRE(S)"
COL_APELLIDO = "APELLIDO(S)"
COL_FECHA_EMI = "FECHA DE EMISIÓN"
COL_FECHA_VERIF = "FECHA DE VERIFICACIÓN"
COL_ESTUDIO = "ESTUDIO/PRÁCTICA/MATERIAL QUE NECESITA"
COL_TIPO = "¿ES PACIENTE O ACOMPAÑANTE?"
COL_VERIFICADOR = "verificador"
COL_LLENADOR = "PERSONA QUE LLENA EL FORMULARIO"

# =========================
# CARGA DE DATOS
# =========================
_df = None
_last_load = 0


def cargar_datos() -> pd.DataFrame:
    try:

        if CSV_URL:
            print("Loading remote CSV data...")

            headers = {
                "User-Agent": (
                    "Mozilla/5.0 (X11; Linux x86_64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/137.0.0.0 Safari/537.36"
                )
            }

            r = requests.get(
                CSV_URL,
                headers=headers,
                timeout=30,
                allow_redirects=True
            )

            print("Status:", r.status_code)
            print("Final URL:", r.url)

            r.raise_for_status()

            df_raw = pd.read_csv(
                StringIO(r.text),
                dtype=str,
                encoding="utf-8-sig",
                engine="python",
                on_bad_lines="warn"
            )

        else:
            print(f"Loading local dataset: {LOCAL_CSV}")

            df_raw = pd.read_csv(
                LOCAL_CSV,
                dtype=str,
                encoding="utf-8-sig"
            )

    except Exception as e:
        print(f"Error reading CSV: {repr(e)}")
        return None

    if "DNI" not in df_raw.columns:
        print("DNI column not found. Columns:", list(df_raw.columns))
        return None

    df_raw["DNI"] = (
        df_raw["DNI"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
    )

    df_raw = df_raw[df_raw["DNI"].str.match(r"^\d{6,10}$")]

    if df_raw.empty:
        print("No valid DNI records found.")
        return None

    print(f"Loaded {len(df_raw)} records.")
    return df_raw


def obtener_datos() -> pd.DataFrame:
    global _df, _last_load
    ahora = time.time()
    if _df is None or (ahora - _last_load) > REFRESH_INTERVAL:
        print("Refreshing dataset...")
        _df = cargar_datos()
        _last_load = ahora
    return _df

def buscar_por_dni(dni: str) -> pd.DataFrame:
    df = obtener_datos()
    if df is None or df.empty:
        return pd.DataFrame()
    dni_limpio = re.sub(r"[^\d]", "", dni.strip())
    return df[df["DNI"] == dni_limpio]

# =========================
# FORMATEO
# =========================
def valor_si_no(val):
    if pd.isna(val):
        return "❌ NO"
    s = str(val).strip().lower()
    return "✅ SÍ" if s in ("si", "sí", "yes", "y", "true", "1") else "❌ NO"

def formatear_registro(row, idx):
    nombre = f"{row.get(COL_NOMBRE, '')} {row.get(COL_APELLIDO, '')}".strip() or "No especificado"
    fecha_emi = row.get(COL_FECHA_EMI, "—")
    fecha_ver = row.get(COL_FECHA_VERIF, "—")
    estudio = row.get(COL_ESTUDIO, "—")
    tipo = row.get(COL_TIPO, "—")
    verificador = row.get(COL_VERIFICADOR, "—")
    llenador = row.get(COL_LLENADOR, "—")

    lineas = [
        f"📌 *Registro {idx+1}*",
        f"👤 *Nombre:* {nombre}",
        f"📅 *Emisión:* {fecha_emi} | *Verif.:* {fecha_ver}",
        "",
        "🔎 *Verificación por sistema:*"
    ]
    for col_reg, col_obs, etiqueta in SISTEMAS:
        reg_val = row.get(col_reg, "")
        obs_val = row.get(col_obs, "")
        estado = valor_si_no(reg_val)
        if "✅" in estado and not pd.isna(obs_val) and str(obs_val).strip():
            lineas.append(f"   {estado} *{etiqueta}*: {obs_val}")
        else:
            lineas.append(f"   {estado} *{etiqueta}*")
    lineas += [
        "",
        "📋 *Información adicional*",
        f"   🧾 *Estudio/material:* {estudio}",
        f"   👥 *Rol:* {tipo}",
        f"   ✍️ *Verificador:* {verificador}",
        f"   📝 *Quién carga:* {llenador}",
        "\n" + "─" * 40 + "\n",
    ]
    return "\n".join(lineas)

def armar_informe(dni: str, df_resultados: pd.DataFrame) -> str:
    if df_resultados.empty:
        return f"⚠️ No se encontraron registros para el DNI *{dni}*."
    encabezado = f"📋 *RESULTADOS DE VERIFICACIÓN PREVISIONAL*\nDNI: *{dni}*\nRegistros encontrados: {len(df_resultados)}\n\n"
    bloques = [formatear_registro(row, i) for i, (_, row) in enumerate(df_resultados.iterrows())]
    texto = encabezado + "\n".join(bloques)
    if len(texto) > 4000:
        texto = texto[:3900] + "\n... (truncado)"
    return texto

# =========================
# HANDLERS
# =========================
estado_usuarios = {}

async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    estado_usuarios[update.effective_user.id] = {"phase": "await_dni"}
    await update.message.reply_text(
        f"🤖 *{BOT_NAME}*\n\n"
        "Ingresá el *DNI* (solo números).\nEjemplo: `28967196`\n\n"
        "Comandos:\n/status - estado de la base de datos\n/cancel - cancelar consulta",
        parse_mode="Markdown"
    )

async def cmd_status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    df = obtener_datos()
    if df is None:
        await update.message.reply_text("❌ Datos no disponibles. Revisá la URL del CSV.")
    else:
        ultima = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(_last_load)) if _last_load else "nunca"
        await update.message.reply_text(
            f"✅ *Estado de la base de datos*\n"
            f"Registros: {len(df)}\n"
            f"Última recarga: {ultima}\n"
            f"Intervalo de refresco: {REFRESH_INTERVAL} segundos",
            parse_mode="Markdown"
        )

async def cmd_cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    estado_usuarios.pop(update.effective_user.id, None)
    await update.message.reply_text("❌ Cancelado. Usá /start.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    uid = update.effective_user.id
    if estado_usuarios.get(uid, {}).get("phase") != "await_dni":
        await update.message.reply_text("Usá /start para iniciar.")
        return
    dni = re.sub(r"[^\d]", "", update.message.text.strip())
    if not re.fullmatch(r"\d{6,10}", dni):
        await update.message.reply_text("❌ DNI inválido (6-10 dígitos).")
        return
    df_res = buscar_por_dni(dni)
    if df_res.empty:
        await update.message.reply_text(f"⚠️ No hay registros para DNI *{dni}*.", parse_mode="Markdown")
        return
    await update.message.reply_text(armar_informe(dni, df_res), parse_mode="Markdown")
    estado_usuarios.pop(uid, None)
    await update.message.reply_text("✅ Finalizada. /start para otro DNI.")

# =========================
# MAIN (sin entrada manual)
# =========================
    
def main():
    print("Cargando datos iniciales...")

    while True:
        df = obtener_datos()
        if df is not None:
            break

        print("Fallo al cargar datos. Reintentando en 30 segundos...")
        time.sleep(30)

    app = Application.builder().token(TELEGRAM_TOKEN).build()

    app.add_handler(CommandHandler("start", cmd_start))
    app.add_handler(CommandHandler("status", cmd_status))
    app.add_handler(CommandHandler("cancel", cmd_cancel))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    print("✅ Bot iniciado. Esperando comandos...")
    app.run_polling()

if __name__ == "__main__":
    main()